# SChain EXPAND

Run a distributed supplier-discovery worker in Colab with shared Neon PostgreSQL state.
Workers using the same experiment ID cooperate safely and can resume after interruption.

In [ ]:
# Run settings
EXPERIMENT_ID = "bmw-supply-chain-v2"
CONFIG_PATH = "config/distributed_neon_deep_discovery.yaml"
RESUME_EXISTING_ONLY = True
APPLY_ONE_TIME_MIGRATION = False
NEW_MAX_TIER = 5
TOP_UP_TIER = 1
TASKS_THIS_SESSION = 40
AUTO_CONTINUE = True
SAVE_TO_NEO4J = False
MODEL = "Qwen/Qwen3-4B-Instruct-2507"
BRANCH = "mirza"

## Install SChain

In [ ]:
import os
import pathlib
import subprocess
import sys

REPO_DIR = pathlib.Path("/content/Schain")
if not (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch",
         "https://github.com/OWNER/SChain.git", str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[postgres]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"], check=True)

## Load secrets

In [ ]:
from google.colab import userdata

required = ["NEON_DATABASE_URL"]
if SAVE_TO_NEO4J:
    required += ["NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"]

for name in required:
    value = userdata.get(name)
    if not value:
        raise RuntimeError(f"Missing Colab secret: {name}")
    os.environ[name] = value

## Start the local model server

In [ ]:
import httpx
import time

VLLM_URL = "http://127.0.0.1:8000/v1"

def vllm_ready() -> bool:
    try:
        return httpx.get(f"{VLLM_URL}/models", timeout=2).status_code == 200
    except httpx.HTTPError:
        return False

if not vllm_ready():
    log = open("/content/vllm.log", "w", encoding="utf-8")
    subprocess.Popen(
        [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
         "--model", MODEL, "--port", "8000"],
        stdout=log,
        stderr=subprocess.STDOUT,
    )
    for _ in range(120):
        if vllm_ready():
            break
        time.sleep(5)
    else:
        raise RuntimeError("vLLM did not become ready; inspect /content/vllm.log")

## Build and inspect the experiment configuration

In [ ]:
from schain.config import load_config

config = load_config(CONFIG_PATH)
config.setdefault("state", {}).update({
    "backend": "postgres",
    "experiment_id": EXPERIMENT_ID,
    "database_url_env": "NEON_DATABASE_URL",
    "resume_existing_only": RESUME_EXISTING_ONLY,
})
config.setdefault("distributed", {})["enabled"] = True
config.setdefault("save", {})["enabled"] = SAVE_TO_NEO4J
config.setdefault("llm", {}).update({"model": MODEL, "base_url": VLLM_URL})

subprocess.run([
    sys.executable, "-m", "schain.distributed.maintenance", "status",
    "--experiment-id", EXPERIMENT_ID, "--json",
], check=False)

## Optional one-time depth extension

Leave `APPLY_ONE_TIME_MIGRATION` disabled for normal runs. Enable it only after taking a
database backup and confirming that no workers or leases are active.

In [ ]:
if APPLY_ONE_TIME_MIGRATION:
    subprocess.run([
        sys.executable, "-m", "schain.distributed.maintenance", "extend",
        "--experiment-id", EXPERIMENT_ID,
        "--new-max-tier", str(NEW_MAX_TIER),
        "--topup-tier", str(TOP_UP_TIER),
        "--confirm",
    ], check=True)

## Run worker sessions

In [ ]:
import json
from schain.distributed.runner import run_distributed_expand

while True:
    result = run_distributed_expand(config, tasks_this_session=TASKS_THIS_SESSION)
    print(json.dumps(result, indent=2, default=str))
    if not AUTO_CONTINUE or result.get("status") in {"completed", "idle"}:
        break

## Export graph and audit artifacts

In [ ]:
from schain.distributed.export import export_distributed_artifacts
from schain.distributed.postgres import PostgresRepository

repository = PostgresRepository.from_config(config)
paths = export_distributed_artifacts(repository, EXPERIMENT_ID, pathlib.Path("outputs"))
print(json.dumps({name: str(path) for name, path in paths.items()}, indent=2))